# Privacy Risk Classifier — Colab Demo

This notebook contains only the project's machine-learning model and dataset workflow. It does not run the web app, API, database, PII extraction service, or OCR pipeline.

**Model input:** seven engineered numeric features (counts/signals derived upstream from text). The classifier does **not** process raw text directly. The bundled dataset is **synthetic**; its scores are demonstration results, not evidence of real-world detection accuracy.

Upload `privacy-risk-colab-bundle.zip` when prompted. It contains the trained model, metadata, and CSV dataset.

In [ ]:
%pip -q install scikit-learn joblib pandas matplotlib

In [ ]:
from google.colab import files
from pathlib import Path
import json
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one privacy-risk-colab-bundle.zip file.')

bundle_dir = Path('/content/privacy_risk_bundle')
bundle_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(bundle_dir)

dataset_path = next(bundle_dir.rglob('privacy_risk_dataset.csv'))
model_path = next(bundle_dir.rglob('privacy_risk_model.joblib'))
metadata_path = next(bundle_dir.rglob('metadata.json'))
print('Dataset:', dataset_path)
print('Trained model:', model_path)
print('Metadata:', metadata_path)

## Dataset and model details

Each row has a synthetic text example, a risk label, and seven numeric features. The label is LOW, MEDIUM, HIGH, or CRITICAL. The saved model is a scikit-learn estimator stored with `joblib`.

In [ ]:
import pandas as pd
import joblib
from IPython.display import display

with metadata_path.open(encoding='utf-8') as f:
    metadata = json.load(f)

df = pd.read_csv(dataset_path)
feature_columns = [
    'finding_count', 'distinct_pii_types', 'authentication_count',
    'financial_count', 'government_id_count', 'context_hits', 'urgency'
]
label_order = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
missing = set(['text', 'risk_level', *feature_columns]) - set(df.columns)
if missing:
    raise ValueError(f'Dataset is missing expected columns: {sorted(missing)}')

print(f"Rows: {len(df):,} | Selected saved model: {metadata['model']} | Synthetic data: {metadata['synthetic']}")
print('\nRisk label counts:')
display(df['risk_level'].value_counts().reindex(label_order, fill_value=0).rename('rows').to_frame())
display(df[['text', 'risk_level', *feature_columns]].head())

## Load and run the full trained model

This is the already-trained project model. Predictions below use feature columns from the dataset; they do not run the upstream text-to-feature detector.

In [ ]:
trained_model = joblib.load(model_path)
demo_rows = df.head(5)
demo_predictions = trained_model.predict(demo_rows[feature_columns])
demo_probabilities = trained_model.predict_proba(demo_rows[feature_columns])

prediction_table = demo_rows[['risk_level']].copy()
prediction_table['model_prediction'] = [label_order[int(value)] for value in demo_predictions]
probability_labels = [label_order[int(value)] for value in trained_model.classes_]
for column_index, label in enumerate(probability_labels):
    prediction_table[f'P({label})'] = demo_probabilities[:, column_index].round(3)
display(prediction_table)

## Reproduce training and compare algorithms

The project compared Logistic Regression, Random Forest, and Gradient Boosting with a stratified 80/20 split. The code below reproduces that evaluation setup and chooses the best estimator by macro F1, using accuracy as the tie-breaker.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = df[feature_columns].astype(int)
label_to_number = {label: number for number, label in enumerate(label_order)}
y = df['risk_level'].map(label_to_number)
if y.isna().any():
    raise ValueError('Dataset contains labels outside LOW/MEDIUM/HIGH/CRITICAL.')
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
models = {
    'logistic_regression': Pipeline([
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=250, max_depth=8, min_samples_leaf=2, random_state=42
    ),
    'gradient_boosting': GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42
    ),
}

results = {}
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    predicted = estimator.predict(X_test)
    results[name] = {
        'estimator': estimator,
        'accuracy': accuracy_score(y_test, predicted),
        'macro_f1': f1_score(y_test, predicted, average='macro', zero_division=0),
        'predicted': predicted,
    }

comparison = pd.DataFrame([
    {'model': name, 'accuracy': result['accuracy'], 'macro_f1': result['macro_f1']}
    for name, result in results.items()
]).sort_values(['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)
display(comparison.style.format({'accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}))

best_name = max(results, key=lambda name: (results[name]['macro_f1'], results[name]['accuracy']))
best_result = results[best_name]
print('Selected model:', best_name)
print('\nConfusion matrix (rows = actual, columns = predicted):')
display(pd.DataFrame(
    confusion_matrix(y_test, best_result['predicted'], labels=range(4)),
    index=[f'actual_{label}' for label in label_order],
    columns=[f'predicted_{label}' for label in label_order],
))
print('\nClassification report:')
print(classification_report(
    y_test, best_result['predicted'], labels=range(4), target_names=label_order, zero_division=0
))

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index('model')[['accuracy', 'macro_f1']].plot(
    kind='bar', ylim=(0, 1), figsize=(8, 4), rot=0, color=['#38bdf8', '#a78bfa']
)
plt.title('Classifier comparison on the synthetic dataset')
plt.ylabel('Score')
plt.xlabel('')
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Export the selected trained model

The exported model predicts numeric class IDs (0–3), corresponding in order to LOW, MEDIUM, HIGH, and CRITICAL. The metadata file records this mapping and the engineered feature order.

In [ ]:
from google.colab import files

output_dir = Path('/content/privacy_risk_model_export')
output_dir.mkdir(exist_ok=True)
export_model_path = output_dir / 'privacy_risk_model.joblib'
export_metadata_path = output_dir / 'metadata.json'
joblib.dump(best_result['estimator'], export_model_path)

export_metadata = {
    'model': best_name,
    'labels': label_order,
    'features': feature_columns,
    'dataset_size': int(len(df)),
    'train_size': int(len(X_train)),
    'test_size': int(len(X_test)),
    'synthetic': True,
    'random_state': 42,
    'accuracy': float(best_result['accuracy']),
    'macro_f1': float(best_result['macro_f1']),
    'warning': 'Evaluation uses synthetic data and does not establish real-world accuracy.',
}
export_metadata_path.write_text(json.dumps(export_metadata, indent=2), encoding='utf-8')

export_zip = Path('/content/privacy_risk_trained_model.zip')
with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(export_model_path, export_model_path.name)
    archive.write(export_metadata_path, export_metadata_path.name)
    archive.write(dataset_path, 'privacy_risk_dataset.csv')
files.download(str(export_zip))